# Daily Streamflow Simulation and Forecasting for the Upper Blue Nile (Abay) Basin, Ethiopia

**Adapted from:** `Hydrological_Modeling_using_GEE_and_LSTM.ipynb` (89-cell source notebook by Kel Markert and Biplov Bhandari).

**Purpose:** Build a reproducible, daily, watershed-scale LSTM workflow for (1) one-day-ahead streamflow simulation/hindcasting and (2) direct multi-day streamflow forecasting using meteorological and land-surface forcing from Google Earth Engine (GEE), combined with partial observed discharge records.

> **Important setup:** The Upper Blue Nile is a large river basin, but a lumped rainfall–runoff model must use the upstream contributing catchment of a particular streamflow gauge—not automatically the entire international basin. This notebook therefore requires the user to provide a verified gauge location and either an upstream basin boundary asset or a HydroBASINS identifier. No Ethiopian gauge coordinates, station identifiers, or discharge values are invented.

### Workflow
1. Configure the gauge, study dates and basin boundary.
2. Export daily ERA5-Land watershed-average forcing from GEE.
3. Load partial observed daily discharge from CSV.
4. align forcing and observations on a continuous daily calendar and diagnose gaps.
5. Train chronological one-day and multi-day LSTM models.
6. Evaluate NSE, KGE, RMSE, MAE and PBIAS, and plot hydrographs and lead-time performance.

**Execution environment:** Google Colab is recommended. Run cells in order. GEE authentication and Drive export require an Earth Engine-enabled Google account.

## 1. Install and import packages

The notebook keeps the original workflow's core elements—Pandas, Earth Engine, TensorFlow/Keras and scikit-learn—but removes the USGS-specific API calls and repairs the original sequence-generation, validation and RMSE issues.

In [ ]:
# Colab: install packages if needed
%pip -q install earthengine-api geemap

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Pandas:", pd.__version__)

## 2. Study configuration

Set the study period to the actual overlap between observed discharge and forcing availability. Because the discharge record is partial, do not assume a fixed 30-year record.

**Basin selection options**
- **Preferred:** upload a verified upstream catchment polygon to your GEE Assets and enter its asset ID.
- **Alternative:** enter the gauge longitude/latitude, inspect candidate HydroBASINS polygons, and set the correct `HYBAS_ID` after checking the outlet and catchment area.

HydroBASINS level 6 is a starting point for candidate discovery, not a guarantee that it matches the gauge's contributing area. Use the most suitable level and validate against official basin/station information.

In [ ]:
# ---------------- USER CONFIGURATION ----------------
STUDY_NAME = "Upper_Blue_Nile_Abay"
START_DATE = "2000-01-01"   # edit to match available records
END_DATE   = "2024-12-31"   # edit to match available records

# Gauge metadata: REQUIRED for reproducibility, not assumed here.
GAUGE_NAME = "ENTER_GAUGE_NAME"
GAUGE_ID = "ENTER_GAUGE_ID"
GAUGE_LAT = None            # e.g., 10.0 -- replace with verified gauge latitude
GAUGE_LON = None            # e.g., 37.0 -- replace with verified gauge longitude

# Basin geometry selection:
BASIN_ASSET_ID = ""         # preferred: users/<username>/<verified_upstream_catchment>
HYDROBASINS_LEVEL = 6      # candidate discovery level: commonly 6, 7 or 8
HYBAS_ID = None             # set only after inspecting candidate polygons

# Observed discharge CSV:
DISCHARGE_CSV_PATH = ""     # leave blank to upload a CSV in Colab
DISCHARGE_DATE_COLUMN = "date"
DISCHARGE_VALUE_COLUMN = "discharge"
DISCHARGE_UNIT = "m3/s"     # supported: "m3/s", "L/s", "cfs"

# GEE export:
GEE_EXPORT_FOLDER = "Abay_LSTM_GEE"
FORCING_EXPORT_NAME = "Abay_ERA5Land_daily_forcing"

# Modeling:
LOOKBACK_DAYS = 90
FORECAST_HORIZON = 7
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
BATCH_SIZE = 64
MAX_EPOCHS = 100
LEARNING_RATE = 1e-3

print("Study:", STUDY_NAME, START_DATE, "to", END_DATE)

## 3. Authenticate and initialize Google Earth Engine

If authentication fails, verify that the Google account has registered for Earth Engine and has access to the selected project. In Colab, select the Cloud project when prompted if required by your account setup.

In [ ]:
import ee

try:
    ee.Initialize()
    print("Earth Engine initialized.")
except Exception:
    ee.Authenticate()
    ee.Initialize()
    print("Earth Engine authenticated and initialized.")

## 4. Define and verify the Upper Blue Nile gauge catchment

### Option A — use a verified catchment asset (recommended)

Upload the delineated upstream catchment polygon to Earth Engine Assets. It should represent the area upstream of the selected discharge gauge.

### Option B — inspect HydroBASINS candidates

If an asset is not available, use a verified gauge point to list candidate polygons. A point may intersect more than one nested basin; inspect the candidate IDs, Pfafstetter attributes and areas before selecting one. Do not simply select the first feature.

In [ ]:
# Choose the basin geometry source.
basin = None
basin_area_km2 = None

if BASIN_ASSET_ID.strip():
    basin_fc = ee.FeatureCollection(BASIN_ASSET_ID)
    basin = basin_fc.geometry()
    basin_area_km2 = basin.area(maxError=100).divide(1e6).getInfo()
    print("Loaded basin asset:", BASIN_ASSET_ID)
    print("Basin area (km2):", round(basin_area_km2, 2))

else:
    if GAUGE_LAT is None or GAUGE_LON is None:
        raise ValueError(
            "Set BASIN_ASSET_ID, or enter verified GAUGE_LAT and GAUGE_LON "
            "to inspect HydroBASINS candidates."
        )

    gauge_point = ee.Geometry.Point([GAUGE_LON, GAUGE_LAT])
    hydro_asset = f"WWF/HydroSHEDS/v1/Basins/hybas_{HYDROBASINS_LEVEL}"
    hydro = ee.FeatureCollection(hydro_asset)
    candidates = hydro.filterBounds(gauge_point)

    # Print candidate metadata for manual selection; do not guess the correct basin.
    candidate_info = candidates.limit(30).getInfo()
    rows = []
    for ft in candidate_info.get("features", []):
        props = ft.get("properties", {})
        rows.append({
            "HYBAS_ID": props.get("HYBAS_ID"),
            "PFAF_ID": props.get("PFAF_ID"),
            "NEXT_DOWN": props.get("NEXT_DOWN"),
            "SUB_AREA": props.get("SUB_AREA"),
            "UP_AREA": props.get("UP_AREA")
        })
    display(pd.DataFrame(rows))

    if HYBAS_ID is None:
        raise ValueError(
            "Review the candidate table and basin outlet. Then set HYBAS_ID "
            "to the verified upstream catchment identifier and rerun this cell."
        )

    selected = hydro.filter(ee.Filter.eq("HYBAS_ID", int(HYBAS_ID)))
    print("Selected feature count:", selected.size().getInfo())
    basin = selected.geometry()
    basin_area_km2 = basin.area(maxError=100).divide(1e6).getInfo()
    print("Selected HydroBASINS ID:", HYBAS_ID)
    print("Basin area (km2):", round(basin_area_km2, 2))

print("Catchment geometry is ready.")

## 5. Visualize the catchment and gauge

Check that the gauge lies at (or very near) the downstream outlet of the selected upstream catchment. A mismatch invalidates the lumped forcing–discharge pairing.

In [ ]:
import geemap

Map = geemap.Map()
Map.centerObject(basin, 7)
Map.addLayer(basin, {"color": "red"}, "Selected upstream catchment")

if GAUGE_LAT is not None and GAUGE_LON is not None:
    gauge_ee = ee.Geometry.Point([GAUGE_LON, GAUGE_LAT])
    Map.addLayer(gauge_ee, {"color": "blue"}, "Streamflow gauge")

Map

## 6. Prepare daily ERA5-Land forcing in GEE

This replaces the source notebook's USGS/Oostanaula-specific and older ERA5 daily extraction with the ERA5-Land Daily Aggregated collection.

Selected predictors:
- precipitation (`total_precipitation_sum`);
- daily minimum and maximum 2 m temperature;
- 10 m wind components;
- surface pressure;
- four soil-water layers;
- total runoff.

ERA5-Land is approximately 11.1 km. The basin mean is a spatially aggregated model/reanalysis estimate, not a point observation. Accumulated water-depth variables are converted from metres to mm/day after export. Temperatures are converted from K to °C.

The code uses per-image `reduceRegion` over the catchment and exports a table to Google Drive. Large date ranges can take time; split long exports into year-wise periods if a task times out.

In [ ]:
ERA5_COLLECTION = "ECMWF/ERA5_LAND/DAILY_AGGR"

ERA5_BANDS = [
    "total_precipitation_sum",
    "temperature_2m_min",
    "temperature_2m_max",
    "u_component_of_wind_10m",
    "v_component_of_wind_10m",
    "surface_pressure",
    "soil_water_layer_1",
    "soil_water_layer_2",
    "soil_water_layer_3",
    "soil_water_layer_4",
    "runoff_sum"
]

era5_daily = (
    ee.ImageCollection(ERA5_COLLECTION)
    .filterDate(START_DATE, ee.Date(END_DATE).advance(1, "day"))
    .select(ERA5_BANDS)
)

print("ERA5-Land image count:", era5_daily.size().getInfo())

In [ ]:
# Reduce each daily image to catchment-average values.
# mean reducer is used for all bands; accumulated bands remain daily totals
# at pixel level before spatial averaging.
def image_to_basin_feature(image):
    image = ee.Image(image)
    stats = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=basin,
        scale=11132,
        maxPixels=1e13,
        tileScale=4
    )
    return ee.Feature(None, stats).set(
        "date", image.date().format("YYYY-MM-dd")
    )

forcing_features = ee.FeatureCollection(
    era5_daily.toList(era5_daily.size()).map(
        lambda image: image_to_basin_feature(ee.Image(image))
    )
)

task = ee.batch.Export.table.toDrive(
    collection=forcing_features,
    description=FORCING_EXPORT_NAME,
    folder=GEE_EXPORT_FOLDER,
    fileNamePrefix=FORCING_EXPORT_NAME,
    fileFormat="CSV"
)
task.start()
print("Export task started. Check the Earth Engine Tasks tab / task status.")
print("Task status:", task.status())

## 7. Download the GEE forcing table from Google Drive

Wait until the Earth Engine export task is **COMPLETED**. Then download the CSV from the configured Drive folder and upload it to this Colab session, or place it at `FORCING_CSV_PATH`.

In [ ]:
FORCING_CSV_PATH = ""  # e.g. "/content/Abay_ERA5Land_daily_forcing.csv"

if not FORCING_CSV_PATH:
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_candidates = [name for name in uploaded if name.lower().endswith(".csv")]
        if len(csv_candidates) == 0:
            raise FileNotFoundError("No CSV uploaded.")
        print("Uploaded files:", csv_candidates)
        # If multiple CSVs were uploaded, select the forcing export explicitly.
        FORCING_CSV_PATH = csv_candidates[0]
    except ImportError:
        raise FileNotFoundError(
            "Set FORCING_CSV_PATH to the downloaded GEE forcing CSV."
        )

forcing_raw = pd.read_csv(FORCING_CSV_PATH)
display(forcing_raw.head())
print(forcing_raw.columns.tolist())

## 8. Clean and convert ERA5-Land forcing

Small negative or implausible accumulated values can occur in packed reanalysis accumulations. The code clips negative precipitation/runoff values to zero, but you should inspect the frequency and magnitude before publication. Do not silently treat this as a substitute for quality control.

ERA5-Land accumulated total evaporation is deliberately not used here because the catalog documents known band-value swaps for three evaporation components. The selected runoff and soil-water bands avoid relying on those affected evaporation components.

In [ ]:
def prepare_era5_forcing(raw):
    raw = raw.copy()
    if "date" not in raw.columns:
        raise ValueError("GEE export must contain a 'date' column.")

    raw["date"] = pd.to_datetime(raw["date"], errors="coerce")
    raw = raw.dropna(subset=["date"]).set_index("date").sort_index()
    raw.index = raw.index.normalize()

    rename = {
        "total_precipitation_sum": "precip_mm",
        "temperature_2m_min": "tmin_c",
        "temperature_2m_max": "tmax_c",
        "u_component_of_wind_10m": "u_wind_ms",
        "v_component_of_wind_10m": "v_wind_ms",
        "surface_pressure": "surface_pressure_pa",
        "soil_water_layer_1": "soil_water_l1",
        "soil_water_layer_2": "soil_water_l2",
        "soil_water_layer_3": "soil_water_l3",
        "soil_water_layer_4": "soil_water_l4",
        "runoff_sum": "runoff_mm"
    }
    missing = [c for c in rename if c not in raw.columns]
    if missing:
        raise ValueError(f"Missing expected ERA5-Land columns: {missing}")

    out = raw[list(rename)].rename(columns=rename).apply(pd.to_numeric, errors="coerce")

    # Unit conversions
    out["precip_mm"] = out["precip_mm"] * 1000.0
    out["runoff_mm"] = out["runoff_mm"] * 1000.0
    out["tmin_c"] = out["tmin_c"] - 273.15
    out["tmax_c"] = out["tmax_c"] - 273.15
    out["surface_pressure_kpa"] = out.pop("surface_pressure_pa") / 1000.0

    # ERA5-Land soil-water layer values are volumetric soil water (m3/m3).
    # Clip only physically impossible negative accumulation artifacts.
    out["precip_mm"] = out["precip_mm"].clip(lower=0)
    out["runoff_mm"] = out["runoff_mm"].clip(lower=0)

    # Duplicate dates, if any, are collapsed using the daily mean.
    out = out.groupby(level=0).mean().sort_index()
    return out

forcing_df = prepare_era5_forcing(forcing_raw)
display(forcing_df.head())
print("Forcing period:", forcing_df.index.min(), "to", forcing_df.index.max())
print("Forcing missing values:\n", forcing_df.isna().sum())

## 9. Load partial observed daily discharge

The expected CSV contains at least two columns:
- a date column (default `date`);
- a discharge column (default `discharge`).

Supported units: m³/s, L/s and cfs. Change the configuration above if your file uses different column names or units. If records contain quality flags, screen them according to the data provider's documentation before modeling. This notebook does **not** invent or interpolate missing discharge values.

In [ ]:
if not DISCHARGE_CSV_PATH:
    try:
        from google.colab import files
        uploaded_obs = files.upload()
        obs_candidates = [name for name in uploaded_obs if name.lower().endswith(".csv")]
        if len(obs_candidates) == 0:
            raise FileNotFoundError("Upload the observed discharge CSV.")
        DISCHARGE_CSV_PATH = obs_candidates[0]
    except ImportError:
        raise FileNotFoundError(
            "Set DISCHARGE_CSV_PATH to your observed discharge CSV."
        )

obs_raw = pd.read_csv(DISCHARGE_CSV_PATH)
display(obs_raw.head())
print("Observed data columns:", obs_raw.columns.tolist())

In [ ]:
def prepare_observed_discharge(raw, date_col, value_col, unit="m3/s"):
    if date_col not in raw.columns or value_col not in raw.columns:
        raise ValueError(
            f"CSV must contain '{date_col}' and '{value_col}'. "
            f"Available columns: {raw.columns.tolist()}"
        )

    obs = raw[[date_col, value_col]].copy()
    obs[date_col] = pd.to_datetime(obs[date_col], errors="coerce")
    obs[value_col] = pd.to_numeric(obs[value_col], errors="coerce")
    obs = obs.dropna(subset=[date_col]).set_index(date_col).sort_index()
    obs.index = obs.index.normalize()

    # Convert discharge to m3/s
    unit = unit.lower().strip()
    if unit == "l/s":
        obs[value_col] = obs[value_col] / 1000.0
    elif unit == "cfs":
        obs[value_col] = obs[value_col] * 0.028316846592
    elif unit != "m3/s":
        raise ValueError("DISCHARGE_UNIT must be 'm3/s', 'L/s' or 'cfs'.")

    obs = obs.rename(columns={value_col: "discharge_m3s"})
    obs.loc[obs["discharge_m3s"] < 0, "discharge_m3s"] = np.nan
    # Duplicate timestamps are averaged; inspect duplicates in the raw source.
    obs = obs.groupby(level=0).mean().sort_index()
    return obs

observed_df = prepare_observed_discharge(
    obs_raw,
    DISCHARGE_DATE_COLUMN,
    DISCHARGE_VALUE_COLUMN,
    DISCHARGE_UNIT
)

display(observed_df.head())
print("Observed records:", observed_df["discharge_m3s"].notna().sum())
print("Observed date range:", observed_df.index.min(), "to", observed_df.index.max())

## 10. Merge forcing and discharge on a continuous daily calendar

This is a key correction to the source notebook. Dropping all missing rows without checking dates can make non-consecutive days appear consecutive to the LSTM. We reindex to a complete daily calendar and retain missing targets as missing.

For the first model run, only complete input windows and fully observed target horizons are used. No target discharge values are interpolated. Forcing gaps are also not automatically filled; investigate and resolve them using a documented procedure if necessary.

In [ ]:
start = max(pd.Timestamp(START_DATE), forcing_df.index.min(), observed_df.index.min())
end = min(pd.Timestamp(END_DATE), forcing_df.index.max(), observed_df.index.max())
if start >= end:
    raise ValueError("No overlapping dates. Check study dates and data ranges.")

daily_index = pd.date_range(start=start, end=end, freq="D", name="date")
model_df = forcing_df.reindex(daily_index).join(
    observed_df.reindex(daily_index), how="left"
)

print("Continuous daily dataframe:", model_df.shape)
print("Period:", model_df.index.min(), "to", model_df.index.max())
display(model_df.head())

In [ ]:
# Data availability and gap diagnostics
availability = pd.DataFrame({
    "non_missing": model_df.notna().sum(),
    "missing": model_df.isna().sum(),
    "missing_percent": 100 * model_df.isna().mean()
})
display(availability)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(model_df.index, model_df["discharge_m3s"], lw=0.7)
ax.set_title("Observed daily discharge availability")
ax.set_ylabel("Discharge (m³/s)")
ax.set_xlabel("Date")
ax.grid(alpha=0.25)
plt.show()

print("Longest consecutive missing-discharge spell (days):")
missing = model_df["discharge_m3s"].isna()
groups = (missing != missing.shift()).cumsum()
missing_runs = missing.groupby(groups).sum()
print(int(missing_runs.max()) if len(missing_runs) else 0)

## 11. Define predictors and inspect their quality

The initial feature set includes precipitation, temperature, wind, pressure, soil water and ERA5-Land runoff. Runoff is a modeled land-surface variable and may be strongly related to the target process; treat it as a predictor experiment, not an independent observation.

For a stricter meteorological-only baseline, set `FEATURE_COLUMNS` to precipitation, temperature, wind and pressure only. Compare feature configurations using the same chronological splits.

In [ ]:
FEATURE_COLUMNS = [
    "precip_mm", "tmin_c", "tmax_c",
    "u_wind_ms", "v_wind_ms", "surface_pressure_kpa",
    "soil_water_l1", "soil_water_l2", "soil_water_l3", "soil_water_l4",
    "runoff_mm"
]
TARGET_COLUMN = "discharge_m3s"

# Optional simpler baseline:
# FEATURE_COLUMNS = [
#     "precip_mm", "tmin_c", "tmax_c",
#     "u_wind_ms", "v_wind_ms", "surface_pressure_kpa"
# ]

assert all(c in model_df.columns for c in FEATURE_COLUMNS + [TARGET_COLUMN])
print("Predictors:", FEATURE_COLUMNS)

## 12. Chronological train/validation/test split

A random split is inappropriate for this time-series task. The following split is chronological:
- earliest 70%: training;
- next 15%: validation;
- final 15%: independent testing.

The boundaries are based on the daily calendar, not on the number of complete windows. This prevents future dates from entering training. Feature and target scalers are fitted only on the training period.

In [ ]:
n_dates = len(model_df)
train_end_pos = int(n_dates * TRAIN_FRACTION) - 1
val_end_pos = int(n_dates * (TRAIN_FRACTION + VALIDATION_FRACTION)) - 1

train_end_date = model_df.index[train_end_pos]
val_end_date = model_df.index[val_end_pos]
test_start_date = model_df.index[val_end_pos + 1]

print("Training through:", train_end_date)
print("Validation:", model_df.index[train_end_pos + 1], "to", val_end_date)
print("Testing:", test_start_date, "to", model_df.index[-1])

## 13. Build leakage-aware LSTM sequences

The original notebook's multi-step sequence code repeated the first target across the forecast horizon. This implementation creates consecutive future targets.

For each sample:
- inputs are the preceding `LOOKBACK_DAYS` of meteorological/land-surface predictors;
- targets are the next `horizon` daily discharge observations;
- the full target horizon must be observed;
- every input predictor value must be available;
- the sample is assigned to a split according to its target dates.

The model does not use observed discharge as an input predictor. Therefore the multi-day model is a **meteorology/land-surface-driven direct forecast conditioned on historical forcing**, not an autoregressive discharge model.

In [ ]:
def make_sequences(df, feature_cols, target_col, lookback, horizon,
                   feature_scaler, target_scaler,
                   train_end, val_end):
    X_raw = df[feature_cols].to_numpy(dtype=np.float32)
    y_raw = df[target_col].to_numpy(dtype=np.float32)
    dates = df.index

    # Fit scalers on training period only.
    train_feature_rows = df.loc[:train_end, feature_cols].dropna()
    train_target_rows = df.loc[:train_end, [target_col]].dropna()

    if len(train_feature_rows) == 0 or len(train_target_rows) == 0:
        raise ValueError("Insufficient non-missing training observations.")

    feature_scaler.fit(train_feature_rows)
    target_scaler.fit(train_target_rows)

    # Transform only rows with complete features; preserve NaNs as NaNs.
    X_scaled = np.full_like(X_raw, np.nan, dtype=np.float32)
    valid_x = np.isfinite(X_raw).all(axis=1)
    X_scaled[valid_x] = feature_scaler.transform(X_raw[valid_x]).astype(np.float32)

    y_scaled = np.full(len(y_raw), np.nan, dtype=np.float32)
    valid_y = np.isfinite(y_raw)
    y_scaled[valid_y] = target_scaler.transform(
        y_raw[valid_y].reshape(-1, 1)
    ).ravel().astype(np.float32)

    split_data = {
        "train": ([], [], []),
        "val": ([], [], []),
        "test": ([], [], [])
    }

    for target_start in range(lookback, len(df) - horizon + 1):
        x_start = target_start - lookback
        x_end = target_start
        y_end = target_start + horizon

        x_window = X_scaled[x_start:x_end]
        y_window = y_scaled[target_start:y_end]

        if not np.isfinite(x_window).all() or not np.isfinite(y_window).all():
            continue

        target_start_date = dates[target_start]
        target_end_date = dates[y_end - 1]

        if target_end_date <= train_end:
            split = "train"
        elif target_start_date > train_end and target_end_date <= val_end:
            split = "val"
        elif target_start_date > val_end:
            split = "test"
        else:
            # Do not allow target windows to straddle split boundaries.
            continue

        split_data[split][0].append(x_window)
        split_data[split][1].append(y_window)
        split_data[split][2].append(target_start_date)

    result = {}
    for split, (xs, ys, ds) in split_data.items():
        if not xs:
            raise ValueError(
                f"No valid {split} sequences. Check data completeness, "
                "lookback, horizon and split dates."
            )
        result[split] = (
            np.asarray(xs, dtype=np.float32),
            np.asarray(ys, dtype=np.float32),
            pd.DatetimeIndex(ds, name="forecast_start_date")
        )
    return result, feature_scaler, target_scaler

## 14. Define the LSTM model and callbacks

The residual connection in the source notebook is retained in a shape-safe form: the first LSTM returns a sequence with the same number of channels as the input, allowing addition to the input tensor. Huber loss is used as a robust baseline. NSE is computed on the full validation/test arrays after prediction rather than optimized independently in mini-batches.

In [ ]:
def build_lstm(input_shape, horizon, units=64, dropout=0.2, learning_rate=1e-3):
    inputs = keras.Input(shape=input_shape, name="meteorological_sequence")

    x = layers.LSTM(
        input_shape[-1],
        return_sequences=True,
        name="lstm_residual_projection"
    )(inputs)

    x = layers.Add(name="residual_add")([inputs, x])

    x = layers.LSTM(units, return_sequences=False, name="lstm_encoder")(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(horizon, activation="linear", name="discharge_forecast")(x)

    model = keras.Model(inputs, outputs, name=f"Abay_LSTM_{horizon}day")
    optimizer = keras.optimizers.Adam(
        learning_rate=learning_rate,
        clipnorm=1.0
    )
    model.compile(
        optimizer=optimizer,
        loss=keras.losses.Huber(),
        metrics=[
            keras.metrics.MeanAbsoluteError(name="mae"),
            keras.metrics.RootMeanSquaredError(name="rmse")
        ]
    )
    return model


def make_callbacks(model_path):
    return [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=12,
            restore_best_weights=True, verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5,
            patience=5, min_lr=1e-6, verbose=1
        ),
        keras.callbacks.ModelCheckpoint(
            model_path, monitor="val_loss",
            save_best_only=True, verbose=1
        )
    ]

## 15. Train and evaluate the one-day-ahead simulation model

This is a one-day-ahead hindcast: the model uses the preceding lookback window of forcing to estimate the next day's discharge. It is not a same-day simulation using observed same-day precipitation.

In [ ]:
def run_experiment(horizon, experiment_name):
    feature_scaler = RobustScaler()
    target_scaler = RobustScaler()

    data, feature_scaler, target_scaler = make_sequences(
        model_df,
        FEATURE_COLUMNS,
        TARGET_COLUMN,
        LOOKBACK_DAYS,
        horizon,
        feature_scaler,
        target_scaler,
        train_end_date,
        val_end_date
    )

    X_train, y_train, d_train = data["train"]
    X_val, y_val, d_val = data["val"]
    X_test, y_test, d_test = data["test"]

    model = build_lstm(
        input_shape=(LOOKBACK_DAYS, len(FEATURE_COLUMNS)),
        horizon=horizon,
        learning_rate=LEARNING_RATE
    )

    print(experiment_name, "shapes:")
    print("Train:", X_train.shape, y_train.shape)
    print("Validation:", X_val.shape, y_val.shape)
    print("Test:", X_test.shape, y_test.shape)

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=False,
        callbacks=make_callbacks(f"{experiment_name}_best.keras"),
        verbose=1
    )

    pred_scaled = model.predict(X_test, verbose=0)

    # Inverse transform each lead-time prediction.
    y_test_original = target_scaler.inverse_transform(
        y_test.reshape(-1, 1)
    ).reshape(y_test.shape)

    pred_original = target_scaler.inverse_transform(
        pred_scaled.reshape(-1, 1)
    ).reshape(pred_scaled.shape)

    return {
        "model": model,
        "history": history,
        "X_test": X_test,
        "y_test": y_test_original,
        "prediction": pred_original,
        "dates": d_test,
        "feature_scaler": feature_scaler,
        "target_scaler": target_scaler,
        "horizon": horizon
    }

simulation_result = run_experiment(1, "Abay_LSTM_1day_simulation")

## 16. Train the direct multi-day forecasting model

The default forecast horizon is seven days. The model outputs all future days simultaneously. This avoids the source notebook's target-broadcasting error.

**Operational limitation:** because future meteorological forcing is not supplied to this model, it predicts streamflow from the historical forcing window only. For operational forecasts, provide forecast meteorology (for example, numerical weather prediction precipitation and temperature) as future inputs and redesign the sequence-to-sequence model accordingly.

In [ ]:
forecast_result = run_experiment(
    FORECAST_HORIZON,
    f"Abay_LSTM_{FORECAST_HORIZON}day_forecast"
)

## 17. Hydrological performance metrics

NSE, KGE, RMSE, MAE, PBIAS and R² are calculated from inverse-transformed discharge in m³/s. For multi-step forecasting, metrics are reported separately for each lead time and pooled across all leads.

KGE here uses the standard correlation, variability ratio and mean ratio components. PBIAS convention used here is `100 * sum(simulated - observed) / sum(observed)`; state this convention in reporting.

In [ ]:
def hydrological_metrics(obs, sim):
    obs = np.asarray(obs, dtype=float).ravel()
    sim = np.asarray(sim, dtype=float).ravel()
    valid = np.isfinite(obs) & np.isfinite(sim)
    obs, sim = obs[valid], sim[valid]

    if len(obs) < 2:
        return {"N": len(obs), "NSE": np.nan, "KGE": np.nan,
                "RMSE": np.nan, "MAE": np.nan, "PBIAS_%": np.nan, "R2": np.nan}

    denom = np.sum((obs - np.mean(obs)) ** 2)
    nse = 1 - np.sum((obs - sim) ** 2) / denom if denom > 0 else np.nan

    r = np.corrcoef(obs, sim)[0, 1] if np.std(obs) > 0 and np.std(sim) > 0 else np.nan
    alpha = np.std(sim, ddof=1) / np.std(obs, ddof=1) if np.std(obs, ddof=1) > 0 else np.nan
    beta = np.mean(sim) / np.mean(obs) if abs(np.mean(obs)) > 1e-12 else np.nan
    kge = (
        1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)
        if np.isfinite([r, alpha, beta]).all() else np.nan
    )

    pbias = 100 * np.sum(sim - obs) / np.sum(obs) if abs(np.sum(obs)) > 1e-12 else np.nan

    return {
        "N": len(obs),
        "NSE": nse,
        "KGE": kge,
        "RMSE": np.sqrt(mean_squared_error(obs, sim)),
        "MAE": mean_absolute_error(obs, sim),
        "PBIAS_%": pbias,
        "R2": r2_score(obs, sim)
    }


def evaluate_experiment(result, label):
    rows = []
    obs = result["y_test"]
    sim = result["prediction"]
    for lead in range(result["horizon"]):
        metrics = hydrological_metrics(obs[:, lead], sim[:, lead])
        metrics["Lead_day"] = lead + 1
        rows.append(metrics)

    pooled = hydrological_metrics(obs, sim)
    pooled["Lead_day"] = "Pooled"
    rows.append(pooled)

    table = pd.DataFrame(rows).set_index("Lead_day")
    print(label)
    display(table.round(4))
    return table

simulation_metrics = evaluate_experiment(
    simulation_result, "One-day-ahead simulation test metrics"
)
forecast_metrics = evaluate_experiment(
    forecast_result, f"{FORECAST_HORIZON}-day direct forecast test metrics"
)

## 18. Plot training history and one-day-ahead hydrograph

In [ ]:
def plot_history(result, title):
    hist = pd.DataFrame(result["history"].history)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(hist["loss"], label="Training")
    ax[0].plot(hist["val_loss"], label="Validation")
    ax[0].set_title("Huber loss")
    ax[0].set_xlabel("Epoch")
    ax[0].legend()
    ax[1].plot(hist["mae"], label="Training MAE")
    ax[1].plot(hist["val_mae"], label="Validation MAE")
    ax[1].set_title("MAE (scaled target)")
    ax[1].set_xlabel("Epoch")
    ax[1].legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_history(simulation_result, "One-day simulation training history")

sim_dates = simulation_result["dates"]
sim_obs = simulation_result["y_test"][:, 0]
sim_pred = simulation_result["prediction"][:, 0]

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(sim_dates, sim_obs, label="Observed", lw=1)
ax.plot(sim_dates, sim_pred, label="LSTM estimate", lw=1)
ax.set_title("Upper Blue Nile: one-day-ahead streamflow hindcast")
ax.set_ylabel("Discharge (m³/s)")
ax.set_xlabel("Date")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## 19. Plot multi-day forecast performance by lead time

In [ ]:
plot_history(forecast_result, f"{FORECAST_HORIZON}-day forecast training history")

lead_metrics = forecast_metrics.drop(index="Pooled", errors="ignore")
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(lead_metrics.index.astype(int), lead_metrics["NSE"], marker="o", label="NSE")
ax.set_xlabel("Forecast lead time (days)")
ax.set_ylabel("NSE")
ax.set_title("Forecast skill versus lead time")
ax.set_xticks(range(1, FORECAST_HORIZON + 1))
ax.grid(alpha=0.25)
plt.show()

# Example: plot the first test forecast window
first_date = forecast_result["dates"][0]
h = forecast_result["horizon"]
dates_window = pd.date_range(first_date, periods=h, freq="D")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(dates_window, forecast_result["y_test"][0], marker="o", label="Observed")
ax.plot(dates_window, forecast_result["prediction"][0], marker="s", label="Forecast")
ax.set_title(f"Example {h}-day forecast starting {first_date.date()}")
ax.set_ylabel("Discharge (m³/s)")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## 20. Save predictions, metrics and models

The CSV outputs preserve forecast start dates and lead times. Keep the feature list, dates, gauge metadata, basin identifier/asset, data source versions and scaler objects with the trained model for reproducibility.

In [ ]:
def export_prediction_table(result, prefix):
    rows = []
    for i, start_date in enumerate(result["dates"]):
        for lead in range(result["horizon"]):
            rows.append({
                "forecast_start_date": start_date,
                "target_date": start_date + pd.Timedelta(days=lead),
                "lead_day": lead + 1,
                "observed_discharge_m3s": result["y_test"][i, lead],
                "predicted_discharge_m3s": result["prediction"][i, lead]
            })
    out = pd.DataFrame(rows)
    out.to_csv(f"{prefix}_test_predictions.csv", index=False)
    return out

simulation_predictions = export_prediction_table(
    simulation_result, "Abay_LSTM_1day"
)
forecast_predictions = export_prediction_table(
    forecast_result, f"Abay_LSTM_{FORECAST_HORIZON}day"
)

simulation_metrics.to_csv("Abay_LSTM_1day_metrics.csv")
forecast_metrics.to_csv(f"Abay_LSTM_{FORECAST_HORIZON}day_metrics.csv")

simulation_result["model"].save("Abay_LSTM_1day.keras")
forecast_result["model"].save(f"Abay_LSTM_{FORECAST_HORIZON}day.keras")

print("Saved models, prediction tables and metric tables.")

## 21. Scientific quality-control checklist before interpreting results

- [ ] Gauge identity, coordinates, datum, units and quality flags verified with the data provider.
- [ ] Selected polygon is the **upstream contributing catchment** for that gauge.
- [ ] Catchment area and outlet location checked against independent basin/station information.
- [ ] GEE export dates, missing pixels and units checked.
- [ ] Observed discharge gaps and duplicate dates documented.
- [ ] No discharge values were fabricated or interpolated to create training targets.
- [ ] Training/validation/test periods are chronological and reported explicitly.
- [ ] Feature and target scalers were fitted on training data only.
- [ ] Forecast skill is reported for each lead time.
- [ ] Baseline models (e.g., persistence, climatology, Random Forest/gradient boosting) are compared.
- [ ] High-flow and low-flow performance are assessed separately.
- [ ] Results are not described as operational forecasts unless future meteorological forecasts are supplied.
- [ ] Sensitivity to lookback length, feature set, basin averaging and missing-data treatment is assessed.

### Important interpretation notes
1. **Partial discharge records:** LSTM performance may be biased if missingness is concentrated in flood seasons or particular years. Report coverage by month/season and the number of valid target sequences.
2. **Spatial representativeness:** ERA5-Land has a grid spacing of about 11.1 km. Small or mountainous catchments may be poorly represented by a simple basin mean.
3. **Runoff predictor:** ERA5-Land runoff is a model-derived variable. Test models both with and without it to understand whether the apparent gain is dependent on a reanalysis hydrological product.
4. **Forecast definition:** A 7-day direct forecast based only on past forcing does not have access to future rainfall. For real forecast operation, add forecast precipitation/temperature from a weather prediction system.
5. **Generalization:** For multiple Abay sub-basins, use a separate spatial holdout (leave-one-catchment-out) experiment; do not randomly mix samples from the same basin across training and test.

## 22. Data sources and citation notes

- **ERA5-Land Daily Aggregated, GEE:** `ECMWF/ERA5_LAND/DAILY_AGGR`. Approximate pixel size 11,132 m. Cite the dataset and Copernicus Climate Change Service information as required by the data license. Google Earth Engine catalog: https://developers.google.com/earth-engine/datasets/catalog/ECMWF_ERA5_LAND_DAILY_AGGR
- **HydroSHEDS / HydroBASINS:** WWF HydroSHEDS basin polygons available in Earth Engine as `WWF/HydroSHEDS/v1/Basins/hybas_6` (and other levels). Validate the selected polygon rather than assuming a level corresponds to a gauge catchment.
- **CHIRPS Daily (optional precipitation comparison):** `UCSB-CHG/CHIRPS/DAILY`, band `precipitation` in mm/day. Catalog: https://developers.google.com/earth-engine/datasets/catalog/UCSB-CHG_CHIRPS_DAILY. CHIRPS may be used as an alternative precipitation predictor after an additional GEE extraction and date merge.

**Methodological continuity with the source notebook:** this version retains the GEE-to-Pandas-to-LSTM workflow, residual LSTM idea, robust scaling and streamflow evaluation objective. It replaces the USGS-only observation retrieval with a user-supplied Ethiopian gauge CSV, replaces the example US watershed with a verified Abay upstream catchment, and corrects time-series gaps, scaling leakage, target sequence construction, validation and RMSE calculation.